# LASSO Feature Selection (LOOCV)

Another simple alternative to `Feature_Weighting.ipynb`. LASSO (L1-penalized logistic regression) naturally zeroes out most gene coefficients, so it picks a small gene set by design. Leave-one-out cross-validated (8 folds) per sex, same pattern as `RandomForest_Feature_Importance.ipynb`.

**Real data guard:** `USE_REAL_DATA` must stay `False` in this AI session. Only set `True` for your own local, offline run (see `CLAUDE.md`).

**Limitations:**
- How many genes get picked depends a lot on one setting, `C` (regularization strength) — Step 2 sweeps a few values so you can pick a sane one instead of guessing.
- Only 8 samples, thousands of genes — treat this as a cross-check, not a replacement for the main pipeline.
- Male and female are kept separate.

In [ ]:
USE_REAL_DATA = True # keep False here. Only set True in your own local run, never inside an AI tool session, per CLAUDE.md.

import time
import pandas as pd

from lasso_selection import run_loocv_lasso, sweep_C

SEXES = ["female", "male"]
EXPECTED_GENE_COUNTS = {"female": 2881, "male": 4257}
DATA_DIR = "restructured"
TOP_N = 10

## Step 1. Load data and check shapes

In [ ]:
data = {}

for sex in SEXES:
    if USE_REAL_DATA:
        df = pd.read_csv(f"{DATA_DIR}/{sex}_expr.csv")
        with open(f"{DATA_DIR}/{sex}_filtered_gene_symbols.txt") as f:
            gene_cols = [line.strip() for line in f if line.strip()]
    else:
        df = pd.read_csv(f"{DATA_DIR}/{sex}_expr_synthetic.csv")
        gene_cols = [c for c in df.columns if c not in ("Sample", "Group", "Label")]

    assert df.shape[0] == 8, f"{sex}: expected 8 samples, got {df.shape[0]}"
    assert len(gene_cols) == EXPECTED_GENE_COUNTS[sex], (
        f"{sex}: expected {EXPECTED_GENE_COUNTS[sex]} genes, got {len(gene_cols)}"
    )
    assert (df["Group"].value_counts() == 4).all(), f"{sex}: expected a 4/4 Group split"
    assert (df["Label"] == (df["Group"] == "C26").astype(int)).all(), (
        f"{sex}: Label does not match Group for some samples"
    )

    data[sex] = {"X": df[gene_cols], "y": df["Label"]}
    print(f"{sex}: OK, {df.shape[0]} samples x {len(gene_cols)} genes, 4/4 Group split, Label matches Group")

## Step 2. Sweep C to pick a sane regularization strength

`C` controls how many genes survive. Too small and the model selects 0 genes (gives up and predicts the majority class); too large and it selects hundreds. Look for a `C` with a small `mean_genes_per_fold` (aim for a couple dozen) and good accuracy, then set `CHOSEN_C` below.

In [ ]:
C_GRID = [0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.5, 0.7, 1.0]

for sex in SEXES:
    print(f"--- {sex} ---")
    print(sweep_C(data[sex]["X"], data[sex]["y"], C_GRID))
    print()

## Step 3. Sanity check on toy data, using the chosen C

Set `CHOSEN_C` from what you saw in Step 2 (0.3 worked on synthetic data — pick your own from the sweep above). One signal gene, four noise genes — the signal gene should be the only one selected.

In [ ]:
import numpy as np

CHOSEN_C = 0.7  # set this from the Step 2 sweep on your data, not left as the synthetic default

toy_y = pd.Series([0, 0, 0, 0, 1, 1, 1, 1])

rng = np.random.default_rng(42)
toy_X = pd.DataFrame({
    "Signal": [5, 8, 10, 12, 120, 125, 130, 135],  # clean SHAM-low / C26-high split
    "Noise1": rng.uniform(0, 150, size=8),
    "Noise2": rng.uniform(0, 150, size=8),
    "Noise3": rng.uniform(0, 150, size=8),
    "Noise4": rng.uniform(0, 150, size=8),
})

toy_summary, toy_accuracy, _ = run_loocv_lasso(toy_X, toy_y, C=CHOSEN_C)
print(toy_summary)
print(f"toy LOOCV accuracy: {toy_accuracy:.0%}")

assert toy_summary.index[0] == "Signal", "Signal gene should rank #1 by selection frequency"
print("PASS — signal gene ranked above all noise genes")

## Step 4. Run LOOCV LASSO on both sexes

Each fold fits LASSO on 7 samples and predicts the 1 held out. `SelectionFreq` is how often a gene survived with a nonzero coefficient across the 8 folds — genes at 1.0 were picked every time.

In [ ]:
summary_results = {}
accuracy_results = {}
fold_detail_results = {}

for sex in SEXES:
    t0 = time.time()
    summary, accuracy, fold_detail = run_loocv_lasso(data[sex]["X"], data[sex]["y"], C=CHOSEN_C)
    elapsed = time.time() - t0

    summary_results[sex] = summary
    accuracy_results[sex] = accuracy
    fold_detail_results[sex] = fold_detail

    n_ever_selected = (summary["SelectionFreq"] > 0).sum()
    print(f"--- {sex} ---  ({elapsed:.1f}s, C={CHOSEN_C})")
    print(f"LOOCV accuracy: {accuracy:.0%} ({fold_detail['correct'].sum()}/{len(fold_detail)})")
    print(f"mean genes selected per fold: {fold_detail['n_genes_selected'].mean():.1f}")
    print(f"genes ever selected across folds: {n_ever_selected}")
    print(f"Top {TOP_N} genes by SelectionFreq:")
    print(summary.head(TOP_N))
    print()

## Step 5. Save results

Saves the full selection-frequency table, per-fold predictions, and a shortlist (genes selected in every fold) for each sex.

In [ ]:
for sex in SEXES:
    summary = summary_results[sex]

    summary_path = f"{DATA_DIR}/{sex}_lasso_selection.csv"
    shortlist_path = f"{DATA_DIR}/{sex}_lasso_shortlist_genes.txt"
    fold_detail_path = f"{DATA_DIR}/{sex}_lasso_loocv_folds.csv"

    summary.to_csv(summary_path, index_label="Gene")
    fold_detail_results[sex].to_csv(fold_detail_path, index=False)

    shortlist = summary.index[summary["SelectionFreq"] == 1.0].tolist()
    with open(shortlist_path, "w") as f:
        f.write("\n".join(shortlist))

    print(f"{sex}: saved {len(summary)}-gene selection table -> {summary_path}")
    print(f"{sex}: saved {len(shortlist)} genes selected in every fold -> {shortlist_path}")
    print(f"{sex}: saved {len(fold_detail_results[sex])}-fold LOOCV detail -> {fold_detail_path}")